# **Imports**

In [ ]:
import os
import yaml
import glob
import pandas as pd

# **Setup**

In [4]:
###################################
# Define Parameters & Settings
###################################

# output directory for deliverables...
output_dir = os.path.join(r'../assets/sgmap-net/classification')
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# directory for experiments...
exp_dir = r'../experiments/sgmap-net/classification'

# **Loss Ablation**

## *Loss Strategies*

In [5]:
#############################################
# Class Balance Schemes for Loss Ablation
#############################################

# path to imbalance metrics
stats_path = r'../assets/data_eda/esv1p1_imbalance_metrics.csv'

# read into df
stats = pd.read_csv(stats_path)


##### inverse class frequency - PyTorch implementation for BCE pos_weight - (N-n_c) / n_c
N = len(glob.glob(r'../data/**/patches/*mask.tif'))
n_c = stats['Frequency (n)']
icf = (N-n_c) / n_c


##### IRLbl - just IRLbl
irlbl = stats['IRLbl']


##### ENS - Cui et al. (1/ENS)
raw_w_c = 1 / stats['N_eff']
sum_w_c = raw_w_c.sum()
normalize_factor = 7 / sum_w_c
final_w_c = raw_w_c * normalize_factor

In [6]:
icf

0      0.059684
1      0.649718
2      1.252905
3      1.843601
4      3.046178
5     20.615331
6    113.881481
Name: Frequency (n), dtype: float64

In [7]:
irlbl

0      1.000000
1      1.556802
2      2.126017
3      2.683443
4      3.818289
5     20.397909
6    108.411111
Name: IRLbl, dtype: float64

In [8]:
final_w_c

0    0.142326
1    0.158953
2    0.180176
3    0.202848
4    0.251584
5    1.007628
6    5.056486
Name: N_eff, dtype: float64

## *Loss Ablation Results*

In [70]:


loss_dir = r'../experiments/sgmap-net/classification/loss'


exp_dirs = glob.glob(f"{loss_dir}/**/*", recursive=False)

ablations = []
for d in exp_dirs:
    loss_name = os.path.dirname(d).split(os.sep)[-1]
    input_name = d.split(os.sep)[-1].split('_')[1]
    config_path = os.path.join(d, "config.yml")
    id_path = os.path.join(d, "id_global.csv")
    cd_path = os.path.join(d, "cd_global.csv")

    pos_weight = pd.NA
    if 'pos_weight' in loss_name:
        pos_weight = loss_name.split('-')[1]

    gamma = pd.NA
    alpha = pd.NA
    with open(config_path, "r") as f:
        c = yaml.safe_load(f)
        if 'gamma' in d or 'alpha' in d:
            gamma = c['loss']['classification']['params']['gamma']
            if 'gamma_' in d:
                pos_weight = loss_name.split('_')[1]
        if 'alpha' in d:
            alpha = c['loss']['classification']['params']['alpha']

    id = pd.read_csv(id_path)['macro_f1'].item()
    cd = pd.read_csv(cd_path)['macro_f1'].item()

    df = pd.DataFrame(
        columns=['input', 'pos_weight', 'gamma', 'alpha', 'f1_id', 'f1_cd'], 
        data=[[input_name, pos_weight, gamma, alpha, id, cd]]
        )
    ablations.append(df)

df = pd.concat(ablations)
df['delta'] = df['f1_id'] - df['f1_cd']
df['delta_rel'] = df['delta'] / df['f1_id'] * 100
df['total'] = df['f1_id'] + df['f1_cd']
df.fillna('-', inplace=True)
df.sort_values(['input', 'total'], ascending=[True, False], inplace=True)
output_path = os.path.join(output_dir, 'loss_ablation.csv')
df.to_csv(output_path, index=False)

df

,input,pos_weight,gamma,alpha,f1_id,f1_cd,delta,delta_rel,total
0,dem,-,0.5,0.5,0.680835,0.535058,0.145777,21.411500,1.215892
0,dem,-,3.0,-,0.666824,0.528058,0.138766,20.809985,1.194882
0,dem,-,0.5,0.25,0.662932,0.526220,0.136713,20.622428,1.189152
0,dem,ens,-,-,0.669437,0.508738,0.160699,24.005120,1.178176
0,dem,-,-,-,0.683138,0.492546,0.190591,27.899390,1.175684
0,dem,-,0.5,-,0.659143,0.516193,0.142950,21.687257,1.175335
0,dem,irlbl,0.5,-,0.665752,0.504897,0.160855,24.161396,1.170649
0,dem,-,1.0,-,0.673774,0.489090,0.184684,27.410419,1.162864
0,dem,-,0.5,0.75,0.663911,0.497343,0.166568,25.088953,1.161254
0,dem,icf,-,-,0.658020,0.488861,0.169159,25.707339,1.146880


In [75]:
df.groupby(['pos_weight', 'gamma', 'alpha'])['total'].mean().sort_values(ascending=False)

pos_weight  gamma  alpha
-           0.5    -        1.232564
            3.0    -        1.229897
            0.5    0.75     1.229644
                   0.5      1.229099
                   0.25     1.222793
            -      -        1.220123
            1.0    -        1.218705
            2.0    -        1.216248
            1.5    -        1.213931
irlbl       0.5    -        1.206862
            -      -        1.206665
ens         -      -        1.193714
icf         -      -        1.167348
Name: total, dtype: float64

# **Pre-trained vs. Random Initialization**

In [85]:

random_init_paths = glob.glob(r'../experiments/sgmap-net/classification/random_init/*')

ablations = []
for d in random_init_paths:
    input_name = os.path.split(d)[-1].split('_')[1]
    config_path = os.path.join(d, 'config.yml')
    id_path = os.path.join(d, 'id_global.csv')
    cd_path = os.path.join(d, 'cd_global.csv')

    with open(config_path, "r") as f:
        c = yaml.safe_load(f)
        pt = c['model']['sgmapnet_params']['pretrained']

    id = pd.read_csv(id_path)['macro_f1'].item()
    cd = pd.read_csv(cd_path)['macro_f1'].item()

    df = pd.DataFrame(columns=['input', 'pretrained', 'f1_id', 'f1_cd'], data=[[input_name, pt, id, cd]])
    ablations.append(df)

df = pd.concat(ablations)
df['delta'] = df['f1_id'] - df['f1_cd']
df['delta_rel'] = df['delta'] / df['f1_id'] * 100
df['total'] = df['f1_id'] + df['f1_cd']
df.sort_values(['input', 'total'], ascending=[True, False], inplace=True)
output_path = os.path.join(output_dir, 'pretrain_ablation.csv')
df.to_csv(output_path, index=False)
df

,input,pretrained,f1_id,f1_cd,delta,delta_rel,total
0,dem,True,0.659143,0.516193,0.142950,21.687257,1.175335
0,dem,False,0.650695,0.478100,0.172595,26.524698,1.128795
0,dem+s-ms,True,0.660962,0.601564,0.059398,8.986570,1.262527
0,dem+s-ms,False,0.662831,0.541501,0.121329,18.304710,1.204332
0,s-5,True,0.632307,0.601038,0.031269,4.945165,1.233345
0,s-5,False,0.646918,0.578546,0.068371,10.568794,1.225464
0,s-ms,True,0.644994,0.614055,0.030939,4.796722,1.259049
0,s-ms,False,0.643869,0.583712,0.060156,9.342959,1.227581


# **Single Feature Experiments**

## *No Attention*

In [116]:

single_dirs = glob.glob(r'../experiments/sgmap-net/classification/single/*')

experiments = []
for d in single_dirs:

    input_name = os.path.split(d)[-1].split('_')[1]

    df = pd.DataFrame()
    df.loc[0, 'input'] = input_name

    id_path = os.path.join(d, 'id_global.csv')
    cd_path = os.path.join(d, 'cd_global.csv')

    if os.path.isfile(id_path):
        id = pd.read_csv(id_path)
        df = pd.merge(left=df, right=id, left_index=True, right_index=True)

    if os.path.isfile(cd_path):
            cd = pd.read_csv(cd_path)
            df = pd.merge(left=df, right=cd, left_index=True, right_index=True, suffixes=['', '_cd'])

    experiments.append(df)

df = pd.concat(experiments)
df['delta_f1'] = df['macro_f1'] - df['macro_f1_cd']
df['delta_rel_f1'] = df['delta_f1'] / df['macro_f1'] * 100
df['total_f1'] = df['macro_f1'] + df['macro_f1_cd']
output_path = r'../assets/sgmap-net/classification/single_global.csv'
df.to_csv(output_path, index=False)

df.sort_values('total_f1', ascending=False).head(10)

,input,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_cd,macro_recall_cd,macro_f1_cd,auroc_cd,macro_map_cd,micro_accuracy_cd,delta_f1,delta_rel_f1,total_f1
0,s-5,0.613144,0.646429,0.627785,0.887185,0.688844,0.870443,0.548635,0.709835,0.613342,0.816513,0.636976,0.860491,0.014443,2.300659,1.241127
0,s-10,0.602556,0.687734,0.633406,0.876993,0.681119,0.867374,0.532449,0.688969,0.594051,0.835626,0.627097,0.855841,0.039356,6.213315,1.227457
0,prc-5,0.608418,0.670317,0.632091,0.879578,0.679950,0.868862,0.524478,0.707077,0.592531,0.818411,0.625153,0.856678,0.039560,6.258567,1.224622
0,prc-10,0.600351,0.710302,0.639904,0.878638,0.678656,0.860677,0.506107,0.711817,0.579141,0.795789,0.605785,0.843843,0.060763,9.495712,1.219045
0,s-20,0.594806,0.661322,0.621944,0.853701,0.668024,0.863095,0.521021,0.678767,0.585107,0.791880,0.613575,0.854725,0.036837,5.922862,1.207052
0,sds-51,0.586235,0.710701,0.632724,0.855533,0.661500,0.851842,0.484454,0.714780,0.566482,0.787103,0.580041,0.822359,0.066242,10.469382,1.199206
0,sds-5,0.595379,0.660785,0.621433,0.867985,0.675169,0.858538,0.503170,0.677791,0.570195,0.787526,0.595611,0.840960,0.051238,8.245125,1.191628
0,sds-21,0.576376,0.673832,0.619217,0.854781,0.662742,0.855376,0.486183,0.712837,0.568279,0.792740,0.597652,0.828590,0.050938,8.226191,1.187496
0,s-50,0.577329,0.748894,0.638612,0.837544,0.655453,0.826730,0.471401,0.690070,0.543214,0.777328,0.549891,0.795480,0.095397,14.938211,1.181826
0,sds-101,0.578479,0.724178,0.625974,0.846127,0.658103,0.847098,0.466257,0.714579,0.549478,0.763278,0.561307,0.806083,0.076495,12.220180,1.175452


## *Self attention*

In [ ]:
single_sa_dirs = glob.glob(r'../experiments/sgmap-net/classification/single_sa/*')

experiments = []
for d in single_sa_dirs:

    input_name = os.path.split(d)[-1].split('_')[1]

    df = pd.DataFrame()
    df.loc[0, 'input'] = input_name

    id_path = os.path.join(d, 'id_global.csv')
    cd_path = os.path.join(d, 'cd_global.csv')

    if os.path.isfile(id_path):
        id = pd.read_csv(id_path)
        df = pd.merge(left=df, right=id, left_index=True, right_index=True)

    if os.path.isfile(cd_path):
            cd = pd.read_csv(cd_path)
            df = pd.merge(left=df, right=cd, left_index=True, right_index=True, suffixes=['', '_cd'])

    experiments.append(df)

df = pd.concat(experiments)
df['delta_f1'] = df['macro_f1'] - df['macro_f1_cd']
df['delta_rel_f1'] = df['delta_f1'] / df['macro_f1'] * 100
df['total_f1'] = df['macro_f1'] + df['macro_f1_cd']
output_path = r'../assets/sgmap-net/classification/single_global.csv'
df.to_csv(output_path, index=False)

df.sort_values('total_f1', ascending=False).head(10)

,input,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_cd,macro_recall_cd,macro_f1_cd,auroc_cd,macro_map_cd,micro_accuracy_cd,delta_f1,delta_rel_f1,total_f1
0,rgb,0.578928,0.662440,0.613094,0.828477,0.652061,0.844959,0.441822,0.666587,0.518985,0.700849,0.486771,0.790272,0.094110,15.349939,1.132079
0,nir,0.563781,0.669562,0.610121,0.819568,0.645484,0.843006,0.444063,0.612673,0.503143,0.738885,0.507900,0.794922,0.106978,17.533914,1.113264
0,dem,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## *Comparison*

In [152]:

single_dirs = glob.glob(r'../experiments/sgmap-net/classification/single/*')

single_sa_dirs = glob.glob(r'../experiments/sgmap-net/classification/single_sa/*')

id_comp = []
cd_comp = []
for d in single_sa_dirs:
    root_path = os.path.dirname(os.path.dirname(d))
    input_name = os.path.split(d)[-1].split('_')[1]
    single_d = glob.glob(f"{root_path}/single/*{input_name}*/")[0]

    s_id_path = os.path.join(single_d, 'id_global.csv')
    sa_id_path = os.path.join(d, 'id_global.csv')

    s_cd_path = os.path.join(single_d, 'cd_global.csv')
    sa_cd_path = os.path.join(d, 'cd_global.csv')

    if os.path.isfile(sa_id_path):
        single_id = pd.read_csv(s_id_path)
        sa_id = pd.read_csv(sa_id_path)

        id = single_id.merge(sa_id, how='left', left_index=True, right_index=True, suffixes=['', '_sa'])
        id['input'] = input_name
        id_comp.append(id)

        single_cd = pd.read_csv(s_cd_path)
        sa_cd = pd.read_csv(sa_cd_path)

        cd = single_cd.merge(sa_cd, how='left', left_index=True, right_index=True, suffixes=['', '_sa'])
        cd['input'] = input_name
        cd_comp.append(cd)

df_id = pd.concat(id_comp)
df_cd = pd.concat(cd_comp)


In [153]:
df_id

,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_sa,macro_recall_sa,macro_f1_sa,auroc_sa,macro_map_sa,micro_accuracy_sa,input
0,0.621140,0.708724,0.651104,0.897432,0.696739,0.869606,0.632110,0.760162,0.671874,0.907449,0.724116,0.876116,dem
0,0.592669,0.710050,0.637072,0.867382,0.684218,0.858166,0.589586,0.735481,0.637275,0.870395,0.679112,0.848772,ep-5
0,0.416624,0.762739,0.518481,0.655306,0.485776,0.668713,0.437148,0.699577,0.507380,0.655709,0.485195,0.691778,nhd
0,0.567432,0.690191,0.602919,0.814086,0.639795,0.813802,0.563781,0.669562,0.610121,0.819568,0.645484,0.843006,nir
0,0.456511,0.827027,0.539367,0.674492,0.495843,0.679501,0.455681,0.831681,0.543590,0.687686,0.503288,0.688895,osm
0,0.582667,0.638289,0.600308,0.824619,0.644383,0.836961,0.578928,0.662440,0.613094,0.828477,0.652061,0.844959,rgb


In [155]:
df_cd

,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_sa,macro_recall_sa,macro_f1_sa,auroc_sa,macro_map_sa,micro_accuracy_sa,input
0,0.474032,0.615418,0.518696,0.746455,0.566523,0.838728,0.513544,0.720419,0.571354,0.787741,0.612639,0.815197,dem
0,0.473441,0.475025,0.440331,0.747032,0.501108,0.829241,0.432161,0.466913,0.419714,0.663412,0.466268,0.822545,ep-5
0,0.335390,0.691007,0.419610,0.555783,0.367346,0.618025,0.328102,0.657700,0.410154,0.576932,0.371704,0.636347,nhd
0,0.456615,0.678431,0.471629,0.644385,0.467110,0.650856,0.444063,0.612673,0.503143,0.738885,0.507900,0.794922,nir
0,0.380555,0.812793,0.465360,0.599453,0.407789,0.579520,0.381247,0.823156,0.471178,0.623595,0.417953,0.595796,osm
0,0.455217,0.681258,0.529227,0.701767,0.508836,0.774089,0.441822,0.666587,0.518985,0.700849,0.486771,0.790272,rgb


In [145]:
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon


In [165]:
result = ttest_rel(df_id['macro_f1'], df_id['macro_f1_sa'], alternative="less")

print(result.statistic)
print(result.pvalue)
print(result.confidence_interval(0.95))

-1.2762925542588686
0.1289596411643983
ConfidenceInterval(low=np.float64(-inf), high=np.float64(0.003287945922384878))


In [162]:
stat, p = wilcoxon(df_id['macro_f1'], df_id['macro_f1_sa'], alternative='less')
print(stat)
print(p)

4.0
0.109375


In [163]:
result = ttest_rel(df_cd['macro_f1'], df_cd['macro_f1_sa'], alternative="less")

print(result.statistic)
print(result.pvalue)
print(result.confidence_interval(0.95))

-0.716290917022327
0.2529319760130687
ConfidenceInterval(low=np.float64(-inf), high=np.float64(0.015011484024378413))


In [164]:
stat, p = wilcoxon(df_cd['macro_f1'], df_cd['macro_f1_sa'], alternative='less')
print(stat)
print(p)

9.0
0.421875


# **Multi-scale Feature Experiments**

## *Stacking*

## *Stacking + Self Attention*

## *Separate Encoders + Concatenation*

## *Separate Encoders + Cross-Attention*

# **Multimodal Experiments**